# 🧠 Exercise: Build a Memory Store That Remembers Users
**O'Reilly Live Training — AI Agent Memory Essentials**  
**Segment 2: Vector Memory with ChromaDB**

---

## What you're building in 10 minutes

A memory pipeline that:
1. **Stores** user preferences into ChromaDB
2. **Retrieves** relevant memories when the user sends a new message
3. **Injects** those memories into a system prompt
4. **Calls** an LLM that responds as if it genuinely knows the user

By the end, your agent will go from **goldfish** (forgets everything) to **personal trainer** (remembers everything that matters).

---

## Setup check
Run this cell first. If it passes, you're ready to go. ✅

In [ ]:
# ✅ Setup check — run this first
!pip install chromadb openai
try:
    import chromadb
    from openai import OpenAI
    print("✅ chromadb imported successfully — version:", chromadb.__version__)
    print("✅ openai imported successfully")
    print("\n🚀 You're ready to build!")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Run: pip install chromadb openai")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

---
## Step 1 — Initialize ChromaDB and Create a Collection

Think of this as **building the filing cabinet** and labeling the first folder.

We're using an **in-memory** client so nothing needs to be installed or configured — it just works locally.

In [ ]:
import chromadb

# In-memory client — no setup needed, perfect for the exercise
# For production you'd use: chromadb.PersistentClient(path="./chroma_store")
client = chromadb.Client()

# Create (or get) a collection — this is our "user_preferences" folder
collection = client.get_or_create_collection(
    name="user_preferences"
)

print("✅ Collection created:", collection.name)
print("📂 Filing cabinet is open and ready.")

✅ Collection created: user_preferences
📂 Filing cabinet is open and ready.


---
## Step 2 — Store Memories for a User

We're filing **preference cards** for user `u42`.

Each card has:
- A **document** (the actual memory text)
- An **id** (unique name for the card)
- **Metadata** (the label — who it belongs to, when it was stored)

ChromaDB auto-embeds the text using its built-in model. No OpenAI call needed for storage.

In [ ]:
# User preferences to store — 5 memory cards for user u42
memories = [
    "User prefers dark mode across all interfaces",
    "User wants concise answers, no long explanations",
    "User is a drone operator based in Charlotte NC",
    "User gets frustrated when responses exceed 3 paragraphs",
    "User prefers bullet points over prose for technical topics"
]

ids = [f"mem_{i:03d}" for i in range(len(memories))]

metadatas = [
    {"user_id": "u42", "category": "ui_preference", "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
    {"user_id": "u42", "category": "profile",       "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
    {"user_id": "u42", "category": "communication", "ts": "2026-06"},
]

# Store all 5 cards in one call
collection.add(
    documents=memories,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Stored {len(memories)} memories for user u42")
print("📇 Cards filed:", ids)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 37.2MiB/s]


✅ Stored 5 memories for user u42
📇 Cards filed: ['mem_000', 'mem_001', 'mem_002', 'mem_003', 'mem_004']


---
## Step 3 — Retrieve Relevant Memories

Now a new message comes in from u42.

We query ChromaDB with that message — it finds the **most semantically relevant** cards.

⚠️ Notice: the query words don't exactly match the stored words. That's the point — **meaning over keywords**.

In [ ]:
# New message from the user
user_message = "How should I configure my dashboard settings?"
user_id = "u42"

# Query ChromaDB — semantic match, not keyword match
results = collection.query(
    query_texts=[user_message],   # ChromaDB embeds this automatically
    n_results=3,                   # Return top 3 most relevant memories
    where={"user_id": user_id}    # Only search THIS user's cards
)

retrieved_memories = results["documents"][0]

print("🔍 Query:", user_message)
print("\n📋 Retrieved memories:")
for i, mem in enumerate(retrieved_memories, 1):
    print(f"  {i}. {mem}")

print("\n💡 Note: we searched for 'dashboard settings' and got back UI + communication preferences")
print("   That's semantic search — matching meaning, not keywords.")

🔍 Query: How should I configure my dashboard settings?

📋 Retrieved memories:
  1. User prefers dark mode across all interfaces
  2. User gets frustrated when responses exceed 3 paragraphs
  3. User wants concise answers, no long explanations

💡 Note: we searched for 'dashboard settings' and got back UI + communication preferences
   That's semantic search — matching meaning, not keywords.


---
## Step 4 — Inject Memories into the System Prompt

This is the **briefing handoff**.

Before the LLM sees the user's question — we slip in the memory cards as context.
The LLM reads the briefing first, then answers.

In [ ]:
def build_prompt(user_msg: str, user_id: str) -> tuple[str, str]:
    """
    The full injection pipeline:
    1. Query ChromaDB for relevant memories
    2. Format them as context
    3. Inject into system prompt
    4. Return (system_prompt, user_message) ready for LLM
    """
    # Step 1: Retrieve
    results = collection.query(
        query_texts=[user_msg],
        n_results=3,
        where={"user_id": user_id}
    )
    memories = results["documents"][0]

    # Step 2: Format
    memory_block = "\n".join(f"- {m}" for m in memories)

    # Step 3: Inject
    system_prompt = f"""You are a helpful AI assistant.

User context from memory:
{memory_block}

Use this context to personalize your response. Do not mention that you have memory — just respond naturally."""

    return system_prompt, user_msg


# Preview what gets sent to the LLM
system_prompt, msg = build_prompt(user_message, user_id)

print("📨 SYSTEM PROMPT (what the LLM reads first):")
print("-" * 50)
print(system_prompt)
print("-" * 50)
print("\n💬 USER MESSAGE:", msg)

📨 SYSTEM PROMPT (what the LLM reads first):
--------------------------------------------------
You are a helpful AI assistant.

User context from memory:
- User prefers dark mode across all interfaces
- User gets frustrated when responses exceed 3 paragraphs
- User wants concise answers, no long explanations

Use this context to personalize your response. Do not mention that you have memory — just respond naturally.
--------------------------------------------------

💬 USER MESSAGE: How should I configure my dashboard settings?


---
## Step 5 — Call the LLM and See the Magic ✨

Now we send both to the LLM.

The LLM was NOT trained on this user's preferences. It has no memory of its own.

But watch how it responds — **as if it genuinely knows u42**.

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# ✅ Reads from Colab Secrets — key never visible on screen
# To set up: click the 🔑 key icon in the left sidebar → Add secret → OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client_llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_agent(user_msg: str, user_id: str) -> str:
    system_prompt, msg = build_prompt(user_msg, user_id)

    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",  # cheap and fast — perfect for live demo
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": msg}
        ],
        max_tokens=300
    )
    return response.choices[0].message.content


# Run it!
print("💬 User:", user_message)
print("\n🤖 Agent (with memory):")
print("-" * 50)
response = ask_agent(user_message, user_id)
print(response)

💬 User: How should I configure my dashboard settings?

🤖 Agent (with memory):
--------------------------------------------------
To configure your dashboard settings effectively, consider the following steps:

1. **Choose a Dark Mode**: If available, switch to dark mode for a more comfortable viewing experience.
2. **Widgets**: Select essential widgets that represent the data you want to monitor, such as KPIs, charts, or activity feeds.
3. **Layout**: Arrange widgets in a way that prioritizes the most important information at the top or center.
4. **Refresh Rate**: Set an appropriate data refresh rate based on how often you need updates. 

These adjustments will help you create a dashboard that’s tailored to your needs.


---
## 🆚 Bonus: See the Difference — With vs Without Memory

Run this cell to see the **exact same question** answered two ways:
- 🐟 **Without memory** — generic goldfish response
- 🧠 **With memory** — personalized personal trainer response

In [ ]:
def ask_agent_no_memory(user_msg: str) -> str:
    """Goldfish mode — no memory, bare system prompt"""
    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user",   "content": user_msg}
        ],
        max_tokens=300
    )
    return response.choices[0].message.content


test_question = "How should I configure my dashboard settings?"

print("🐟 WITHOUT MEMORY (goldfish):")
print("-" * 50)
print(ask_agent_no_memory(test_question))

print("\n" + "=" * 50 + "\n")

print("🧠 WITH MEMORY (personal trainer):")
print("-" * 50)
print(ask_agent(test_question, "u42"))

print("\n\n💡 Same question. Same LLM. Completely different response.")
print("   The only difference: we briefed the trainer before they walked in.")

🐟 WITHOUT MEMORY (goldfish):
--------------------------------------------------
Configuring your dashboard settings effectively depends on your specific needs, the platform you are using, and the type of data you want to visualize. Here are some general steps and considerations to help you set up your dashboard:

### 1. Define Your Goals
- **Identify Key Metrics:** Determine what you want to track based on your objectives. This could be sales numbers, website traffic, user engagement, etc.
- **Target Audience:** Understand who will be using the dashboard and what information is most relevant to them.

### 2. Choose the Right Tools
- **Dashboard Software:** Select a dashboard tool that suits your needs (e.g., Tableau, Power BI, Google Data Studio, etc.).
- **Integration Capabilities:** Ensure that the tool can integrate with your data sources (e.g., databases, spreadsheets, APIs).

### 3. Data Sources & Connections
- **Connect Data Sources:** Link your dashboard to the necessary databas

---
## 🎯 Challenge: Add Your Own Memories

Try these before the segment ends:

**Challenge 1 — Add a new memory card:**
```python
collection.add(
    documents=["User is preparing for a FAA security audit in July"],
    ids=["mem_005"],
    metadatas=[{"user_id": "u42", "category": "context", "ts": "2026-06"}]
)
```
Then ask: *"What should I prioritize this month?"* — does the response change?

**Challenge 2 — Try a second user:**  
Store memories for `user_id: "u99"` with completely different preferences.  
Ask the same question for both users. Watch the agent respond differently to each.

**Challenge 3 — Break the keyword assumption:**  
Store: *"User dislikes verbose explanations"*  
Query: *"How do you communicate?"*  
ChromaDB should still find it. Meaning over keywords. ✅

---

## ✅ What you just built

| Step | What happened |
|------|---------------|
| `collection.add()` | Filed memory cards with labels |
| `collection.query()` | Found relevant cards by meaning, not keywords |
| `build_prompt()` | Briefed the LLM before the conversation started |
| `ask_agent()` | Full pipeline: retrieve → inject → respond |

**This is the foundation of every production memory layer.**  
Segment 3 adds conversation patterns on top of this.  
Segment 4 replaces the manual wiring with Mem0 and Zep.  

But the core loop? Always the same three steps: **embed → persist → query.** 🎯